In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

FACTOR_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "11_sp500_monthly_factor_scores_full_2005_2025.parquet"
)

TABLE_OUTPUT_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
)

FIGURE_OUTPUT_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "figures"
)

TABLE_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FIGURE_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print(f"Factor file exists: {FACTOR_FILE.exists()}")
print(f"Factor file: {FACTOR_FILE}")

if not FACTOR_FILE.exists():
    raise FileNotFoundError(
        f"Factor file not found: {FACTOR_FILE}"
    )

In [ ]:
factor_test_df = pd.read_parquet(
    FACTOR_FILE
)

factor_test_df["month"] = pd.to_datetime(
    factor_test_df["month"]
)

factor_test_df = factor_test_df.sort_values(
    [
        "month",
        "permno",
    ]
).reset_index(drop=True)

print("Factor-testing dataset loaded successfully.")
print(f"Number of rows: {len(factor_test_df):,}")
print(f"Number of columns: {factor_test_df.shape[1]}")
print(
    "Date range: "
    f"{factor_test_df['month'].min().date()} "
    f"to {factor_test_df['month'].max().date()}"
)

In [ ]:
duplicate_count = factor_test_df.duplicated(
    subset=[
        "permno",
        "month",
    ]
).sum()

non_member_count = (
    ~factor_test_df["is_member_month_end"]
    .fillna(False)
).sum()

print("Input validation:")
print(f"Duplicate PERMNO-month rows: {duplicate_count:,}")
print(f"Non-member month-end rows: {non_member_count:,}")

if duplicate_count != 0:
    raise ValueError(
        "Duplicate PERMNO-month rows were detected."
    )

if non_member_count != 0:
    raise ValueError(
        "The factor file still contains non-member "
        "month-end observations."
    )

In [ ]:
FACTOR_COLUMNS = [
    "factor_value",
    "factor_momentum",
    "factor_quality",
    "factor_investment",
    "factor_size",
    "factor_low_volatility",
    "multi_factor_score",
]

TARGET_RETURN = "future_return_1m"

MINIMUM_STOCKS_PER_MONTH = 30

print("Factors included in the IC analysis:")

for factor_name in FACTOR_COLUMNS:
    print(f"- {factor_name}")

In [ ]:
monthly_ic_records = []

for factor_name in FACTOR_COLUMNS:

    for month_value, month_group in factor_test_df.groupby(
        "month",
        sort=True,
    ):
        monthly_sample = (
            month_group[
                [
                    factor_name,
                    TARGET_RETURN,
                ]
            ]
            .replace(
                [np.inf, -np.inf],
                np.nan,
            )
            .dropna()
        )

        number_of_stocks = len(
            monthly_sample
        )

        if (
            number_of_stocks
            < MINIMUM_STOCKS_PER_MONTH
        ):
            continue

        if (
            monthly_sample[factor_name].nunique()
            < 2
        ):
            continue

        if (
            monthly_sample[TARGET_RETURN].nunique()
            < 2
        ):
            continue

        pearson_ic = monthly_sample[
            factor_name
        ].corr(
            monthly_sample[TARGET_RETURN]
        )

        factor_rank = monthly_sample[
            factor_name
        ].rank(
            method="average"
        )

        return_rank = monthly_sample[
            TARGET_RETURN
        ].rank(
            method="average"
        )

        rank_ic = factor_rank.corr(
            return_rank
        )

        monthly_ic_records.append({
            "month": month_value,
            "factor": factor_name,
            "number_of_stocks": number_of_stocks,
            "pearson_ic": pearson_ic,
            "rank_ic": rank_ic,
        })

monthly_ic_df = pd.DataFrame(
    monthly_ic_records
)

monthly_ic_df = monthly_ic_df.sort_values(
    [
        "factor",
        "month",
    ]
).reset_index(drop=True)

print("Monthly IC calculations completed.")
print(f"Number of IC records: {len(monthly_ic_df):,}")
print(
    "Number of tested months: "
    f"{monthly_ic_df['month'].nunique()}"
)

In [ ]:
def summarize_ic_series(
    ic_series,
):
    clean_series = (
        ic_series
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .dropna()
    )

    number_of_months = len(
        clean_series
    )

    mean_ic = clean_series.mean()
    median_ic = clean_series.median()
    standard_deviation = clean_series.std(ddof=1)

    if (
        number_of_months > 1
        and standard_deviation > 0
    ):
        monthly_icir = (
            mean_ic
            / standard_deviation
        )

        annualized_icir = (
            monthly_icir
            * np.sqrt(12)
        )

        t_statistic = (
            mean_ic
            / (
                standard_deviation
                / np.sqrt(number_of_months)
            )
        )

        p_value = (
            2
            * stats.t.sf(
                np.abs(t_statistic),
                df=number_of_months - 1,
            )
        )
    else:
        monthly_icir = np.nan
        annualized_icir = np.nan
        t_statistic = np.nan
        p_value = np.nan

    positive_month_rate = (
        clean_series > 0
    ).mean()

    return {
        "number_of_months": number_of_months,
        "mean_ic": mean_ic,
        "median_ic": median_ic,
        "std_ic": standard_deviation,
        "monthly_icir": monthly_icir,
        "annualized_icir": annualized_icir,
        "t_statistic": t_statistic,
        "p_value": p_value,
        "positive_month_rate": positive_month_rate,
    }

In [ ]:
ic_summary_records = []

for factor_name in FACTOR_COLUMNS:

    factor_ic_data = monthly_ic_df.loc[
        monthly_ic_df["factor"]
        == factor_name
    ]

    for ic_column, ic_label in [
        (
            "pearson_ic",
            "Pearson IC",
        ),
        (
            "rank_ic",
            "Rank IC",
        ),
    ]:
        summary_result = summarize_ic_series(
            factor_ic_data[ic_column]
        )

        summary_result["factor"] = factor_name
        summary_result["ic_type"] = ic_label

        ic_summary_records.append(
            summary_result
        )

ic_summary_df = pd.DataFrame(
    ic_summary_records
)

ic_summary_df = ic_summary_df[
    [
        "factor",
        "ic_type",
        "number_of_months",
        "mean_ic",
        "median_ic",
        "std_ic",
        "monthly_icir",
        "annualized_icir",
        "t_statistic",
        "p_value",
        "positive_month_rate",
    ]
]

print("IC summary completed.")

In [ ]:
rank_ic_summary = (
    ic_summary_df.loc[
        ic_summary_df["ic_type"]
        == "Rank IC",
        [
            "factor",
            "number_of_months",
            "mean_ic",
            "annualized_icir",
            "t_statistic",
            "p_value",
            "positive_month_rate",
        ],
    ]
    .copy()
)

pearson_ic_summary = (
    ic_summary_df.loc[
        ic_summary_df["ic_type"]
        == "Pearson IC",
        [
            "factor",
            "number_of_months",
            "mean_ic",
            "annualized_icir",
            "t_statistic",
            "p_value",
            "positive_month_rate",
        ],
    ]
    .copy()
)

print("Rank IC summary:")
print(
    rank_ic_summary
    .round(4)
    .to_string(index=False)
)

print("\nPearson IC summary:")
print(
    pearson_ic_summary
    .round(4)
    .to_string(index=False)
)

MONTHLY_IC_OUTPUT_FILE = (
    TABLE_OUTPUT_DIR
    / "01_monthly_factor_ic.csv"
)

IC_SUMMARY_OUTPUT_FILE = (
    TABLE_OUTPUT_DIR
    / "02_factor_ic_summary.csv"
)

monthly_ic_df.to_csv(
    MONTHLY_IC_OUTPUT_FILE,
    index=False,
)

ic_summary_df.to_csv(
    IC_SUMMARY_OUTPUT_FILE,
    index=False,
)

print("\nIC result files saved successfully.")
print(f"Monthly IC file: {MONTHLY_IC_OUTPUT_FILE}")
print(f"IC summary file: {IC_SUMMARY_OUTPUT_FILE}")

In [ ]:
quintile_return_records = []

for factor_name in FACTOR_COLUMNS:

    for month_value, month_group in factor_test_df.groupby(
        "month",
        sort=True,
    ):
        monthly_sample = (
            month_group[
                [
                    factor_name,
                    TARGET_RETURN,
                    "month_end_market_cap",
                ]
            ]
            .replace(
                [np.inf, -np.inf],
                np.nan,
            )
            .dropna(
                subset=[
                    factor_name,
                    TARGET_RETURN,
                ]
            )
            .copy()
        )

        if len(monthly_sample) < 50:
            continue

        if monthly_sample[factor_name].nunique() < 5:
            continue

        factor_rank = monthly_sample[
            factor_name
        ].rank(
            method="first"
        )

        monthly_sample["quintile"] = (
            pd.qcut(
                factor_rank,
                q=5,
                labels=False,
            )
            + 1
        )

        for quintile_number in range(1, 6):

            quintile_sample = monthly_sample.loc[
                monthly_sample["quintile"]
                == quintile_number
            ]

            equal_weight_return = (
                quintile_sample[
                    TARGET_RETURN
                ].mean()
            )

            valid_weight_sample = (
                quintile_sample.loc[
                    quintile_sample[
                        "month_end_market_cap"
                    ].notna()
                    & (
                        quintile_sample[
                            "month_end_market_cap"
                        ]
                        > 0
                    )
                ]
            )

            if len(valid_weight_sample) > 0:
                value_weight_return = np.average(
                    valid_weight_sample[
                        TARGET_RETURN
                    ],
                    weights=valid_weight_sample[
                        "month_end_market_cap"
                    ],
                )
            else:
                value_weight_return = np.nan

            quintile_return_records.append({
                "month": month_value,
                "factor": factor_name,
                "quintile": quintile_number,
                "number_of_stocks": len(
                    quintile_sample
                ),
                "equal_weight_return": (
                    equal_weight_return
                ),
                "value_weight_return": (
                    value_weight_return
                ),
            })

quintile_returns_df = pd.DataFrame(
    quintile_return_records
)

quintile_returns_df = (
    quintile_returns_df
    .sort_values([
        "factor",
        "month",
        "quintile",
    ])
    .reset_index(drop=True)
)

print("Monthly quintile portfolios created successfully.")
print(
    f"Number of portfolio-month records: "
    f"{len(quintile_returns_df):,}"
)

In [ ]:
quintile_summary_df = (
    quintile_returns_df
    .groupby(
        [
            "factor",
            "quintile",
        ],
        as_index=False,
    )
    .agg(
        number_of_months=(
            "month",
            "nunique",
        ),
        average_number_of_stocks=(
            "number_of_stocks",
            "mean",
        ),
        mean_equal_weight_return=(
            "equal_weight_return",
            "mean",
        ),
        std_equal_weight_return=(
            "equal_weight_return",
            "std",
        ),
        mean_value_weight_return=(
            "value_weight_return",
            "mean",
        ),
        std_value_weight_return=(
            "value_weight_return",
            "std",
        ),
    )
)

quintile_summary_df[
    "annualized_equal_weight_return"
] = (
    quintile_summary_df[
        "mean_equal_weight_return"
    ] * 12
)

quintile_summary_df[
    "annualized_value_weight_return"
] = (
    quintile_summary_df[
        "mean_value_weight_return"
    ] * 12
)

equal_weight_quintile_table = (
    quintile_summary_df
    .pivot(
        index="factor",
        columns="quintile",
        values="mean_equal_weight_return",
    )
    .rename(
        columns={
            1: "Q1",
            2: "Q2",
            3: "Q3",
            4: "Q4",
            5: "Q5",
        }
    )
)

equal_weight_quintile_table[
    "Q5_minus_Q1"
] = (
    equal_weight_quintile_table["Q5"]
    - equal_weight_quintile_table["Q1"]
)

print("Average monthly equal-weight returns (%):")
print(
    (
        equal_weight_quintile_table
        * 100
    )
    .round(3)
    .to_string()
)

In [ ]:
equal_weight_pivot = (
    quintile_returns_df
    .pivot_table(
        index=[
            "month",
            "factor",
        ],
        columns="quintile",
        values="equal_weight_return",
    )
    .reset_index()
)

value_weight_pivot = (
    quintile_returns_df
    .pivot_table(
        index=[
            "month",
            "factor",
        ],
        columns="quintile",
        values="value_weight_return",
    )
    .reset_index()
)

equal_weight_pivot[
    "long_short_return"
] = (
    equal_weight_pivot[5]
    - equal_weight_pivot[1]
)

value_weight_pivot[
    "long_short_return"
] = (
    value_weight_pivot[5]
    - value_weight_pivot[1]
)

print("Long-short return series created.")

In [ ]:
import statsmodels.api as sm


def summarize_long_short_returns(
    return_series,
    maximum_lags=3,
):
    clean_returns = (
        pd.Series(return_series)
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .dropna()
        .astype(float)
    )

    number_of_months = len(
        clean_returns
    )

    mean_monthly_return = (
        clean_returns.mean()
    )

    monthly_volatility = (
        clean_returns.std(ddof=1)
    )

    annualized_return = (
        mean_monthly_return * 12
    )

    annualized_volatility = (
        monthly_volatility
        * np.sqrt(12)
    )

    if monthly_volatility > 0:
        annualized_sharpe = (
            mean_monthly_return
            / monthly_volatility
            * np.sqrt(12)
        )
    else:
        annualized_sharpe = np.nan

    regression_model = sm.OLS(
        clean_returns.to_numpy(),
        np.ones(
            (
                number_of_months,
                1,
            )
        ),
    ).fit(
        cov_type="HAC",
        cov_kwds={
            "maxlags": maximum_lags,
        },
    )

    return {
        "number_of_months": number_of_months,
        "mean_monthly_return": mean_monthly_return,
        "annualized_return": annualized_return,
        "annualized_volatility": annualized_volatility,
        "annualized_sharpe": annualized_sharpe,
        "newey_west_t_statistic": (
            regression_model.tvalues[0]
        ),
        "newey_west_p_value": (
            regression_model.pvalues[0]
        ),
        "positive_month_rate": (
            clean_returns > 0
        ).mean(),
    }

In [ ]:
long_short_summary_records = []

for factor_name in FACTOR_COLUMNS:

    equal_weight_series = (
        equal_weight_pivot.loc[
            equal_weight_pivot["factor"]
            == factor_name,
            "long_short_return",
        ]
    )

    value_weight_series = (
        value_weight_pivot.loc[
            value_weight_pivot["factor"]
            == factor_name,
            "long_short_return",
        ]
    )

    for weighting_method, return_series in [
        (
            "Equal Weight",
            equal_weight_series,
        ),
        (
            "Value Weight",
            value_weight_series,
        ),
    ]:
        summary_result = (
            summarize_long_short_returns(
                return_series
            )
        )

        summary_result["factor"] = (
            factor_name
        )

        summary_result["weighting_method"] = (
            weighting_method
        )

        long_short_summary_records.append(
            summary_result
        )

long_short_summary_df = pd.DataFrame(
    long_short_summary_records
)

long_short_summary_df = long_short_summary_df[
    [
        "factor",
        "weighting_method",
        "number_of_months",
        "mean_monthly_return",
        "annualized_return",
        "annualized_volatility",
        "annualized_sharpe",
        "newey_west_t_statistic",
        "newey_west_p_value",
        "positive_month_rate",
    ]
]

print("Long-short portfolio summary:")
print(
    long_short_summary_df
    .round(4)
    .to_string(index=False)
)

In [ ]:
monotonicity_records = []

for factor_name in FACTOR_COLUMNS:

    factor_quintiles = (
        quintile_summary_df.loc[
            quintile_summary_df["factor"]
            == factor_name
        ]
        .sort_values("quintile")
    )

    quintile_numbers = (
        factor_quintiles[
            "quintile"
        ].to_numpy()
    )

    quintile_returns = (
        factor_quintiles[
            "mean_equal_weight_return"
        ].to_numpy()
    )

    adjacent_increases = (
        np.diff(quintile_returns) > 0
    ).sum()

    monotonicity_correlation = (
        np.corrcoef(
            quintile_numbers,
            quintile_returns,
        )[0, 1]
    )

    monotonicity_records.append({
        "factor": factor_name,
        "adjacent_increases_out_of_four": (
            adjacent_increases
        ),
        "monotonicity_correlation": (
            monotonicity_correlation
        ),
        "q5_minus_q1_average_return": (
            quintile_returns[-1]
            - quintile_returns[0]
        ),
    })

monotonicity_df = pd.DataFrame(
    monotonicity_records
)

print("Factor monotonicity summary:")
print(
    monotonicity_df
    .round(4)
    .to_string(index=False)
)

In [ ]:
quintile_returns_df.to_csv(
    TABLE_OUTPUT_DIR
    / "03_monthly_factor_quintile_returns.csv",
    index=False,
)

quintile_summary_df.to_csv(
    TABLE_OUTPUT_DIR
    / "04_factor_quintile_summary.csv",
    index=False,
)

long_short_summary_df.to_csv(
    TABLE_OUTPUT_DIR
    / "05_factor_long_short_summary.csv",
    index=False,
)

monotonicity_df.to_csv(
    TABLE_OUTPUT_DIR
    / "06_factor_monotonicity_summary.csv",
    index=False,
)

print("Factor-sorting result files saved successfully.")

In [ ]:
REGRESSION_FACTORS = [
    "factor_value",
    "factor_momentum",
    "factor_quality",
    "factor_investment",
    "factor_size",
    "factor_low_volatility",
]

monthly_correlation_matrices = []

for month_value, month_group in factor_test_df.groupby(
    "month",
    sort=True,
):
    monthly_correlation = (
        month_group[
            REGRESSION_FACTORS
        ]
        .corr(
            method="spearman",
            min_periods=30,
        )
    )

    monthly_correlation_matrices.append(
        monthly_correlation.to_numpy()
    )

average_correlation_array = np.nanmean(
    np.stack(
        monthly_correlation_matrices
    ),
    axis=0,
)

average_factor_correlation_df = pd.DataFrame(
    average_correlation_array,
    index=REGRESSION_FACTORS,
    columns=REGRESSION_FACTORS,
)

print("Average monthly Spearman factor correlation:")
print(
    average_factor_correlation_df
    .round(3)
    .to_string()
)

average_factor_correlation_df.to_csv(
    TABLE_OUTPUT_DIR
    / "07_average_factor_correlation.csv"
)

In [ ]:
factor_test_df[
    "future_return_1m_winsorized"
] = (
    factor_test_df
    .groupby("month")[
        TARGET_RETURN
    ]
    .transform(
        lambda monthly_returns: (
            monthly_returns.clip(
                lower=monthly_returns.quantile(
                    0.01
                ),
                upper=monthly_returns.quantile(
                    0.99
                ),
            )
        )
    )
)

print("Winsorized one-month-ahead returns created.")

In [ ]:
def run_monthly_cross_sectional_regressions(
    data,
    target_column,
    factor_columns,
    minimum_observations=100,
):
    monthly_coefficient_records = []

    for month_value, month_group in data.groupby(
        "month",
        sort=True,
    ):
        regression_sample = (
            month_group[
                [
                    target_column,
                    *factor_columns,
                ]
            ]
            .replace(
                [np.inf, -np.inf],
                np.nan,
            )
            .dropna()
        )

        number_of_observations = len(
            regression_sample
        )

        if (
            number_of_observations
            < minimum_observations
        ):
            continue

        dependent_variable = (
            regression_sample[
                target_column
            ].astype(float)
        )

        independent_variables = (
            regression_sample[
                factor_columns
            ].astype(float)
        )

        independent_variables = sm.add_constant(
            independent_variables,
            has_constant="add",
        )

        regression_result = sm.OLS(
            dependent_variable,
            independent_variables,
        ).fit()

        monthly_record = {
            "month": month_value,
            "number_of_observations": (
                number_of_observations
            ),
            "r_squared": (
                regression_result.rsquared
            ),
            "adjusted_r_squared": (
                regression_result.rsquared_adj
            ),
        }

        for coefficient_name in [
            "const",
            *factor_columns,
        ]:
            monthly_record[
                coefficient_name
            ] = regression_result.params.get(
                coefficient_name,
                np.nan,
            )

        monthly_coefficient_records.append(
            monthly_record
        )

    return (
        pd.DataFrame(
            monthly_coefficient_records
        )
        .sort_values("month")
        .reset_index(drop=True)
    )

In [ ]:
def summarize_fama_macbeth_coefficients(
    coefficient_data,
    factor_columns,
    specification_name,
    maximum_lags=3,
):
    summary_records = []

    for variable_name in [
        "const",
        *factor_columns,
    ]:
        coefficient_series = (
            coefficient_data[
                variable_name
            ]
            .replace(
                [np.inf, -np.inf],
                np.nan,
            )
            .dropna()
            .astype(float)
        )

        number_of_months = len(
            coefficient_series
        )

        mean_coefficient = (
            coefficient_series.mean()
        )

        standard_deviation = (
            coefficient_series.std(ddof=1)
        )

        hac_regression = sm.OLS(
            coefficient_series.to_numpy(),
            np.ones(
                (
                    number_of_months,
                    1,
                )
            ),
        ).fit(
            cov_type="HAC",
            cov_kwds={
                "maxlags": maximum_lags,
            },
        )

        summary_records.append({
            "specification": specification_name,
            "variable": variable_name,
            "number_of_months": number_of_months,
            "mean_monthly_coefficient": (
                mean_coefficient
            ),
            "annualized_coefficient": (
                mean_coefficient * 12
            ),
            "coefficient_std": (
                standard_deviation
            ),
            "newey_west_t_statistic": (
                hac_regression.tvalues[0]
            ),
            "newey_west_p_value": (
                hac_regression.pvalues[0]
            ),
            "positive_month_rate": (
                coefficient_series > 0
            ).mean(),
        })

    return pd.DataFrame(
        summary_records
    )

In [ ]:
fmb_raw_coefficients_df = (
    run_monthly_cross_sectional_regressions(
        data=factor_test_df,
        target_column=TARGET_RETURN,
        factor_columns=REGRESSION_FACTORS,
    )
)

fmb_winsorized_coefficients_df = (
    run_monthly_cross_sectional_regressions(
        data=factor_test_df,
        target_column=(
            "future_return_1m_winsorized"
        ),
        factor_columns=REGRESSION_FACTORS,
    )
)

fmb_raw_summary_df = (
    summarize_fama_macbeth_coefficients(
        coefficient_data=(
            fmb_raw_coefficients_df
        ),
        factor_columns=REGRESSION_FACTORS,
        specification_name=(
            "Raw Future Return"
        ),
    )
)

fmb_winsorized_summary_df = (
    summarize_fama_macbeth_coefficients(
        coefficient_data=(
            fmb_winsorized_coefficients_df
        ),
        factor_columns=REGRESSION_FACTORS,
        specification_name=(
            "Winsorized Future Return"
        ),
    )
)

fmb_summary_df = pd.concat(
    [
        fmb_raw_summary_df,
        fmb_winsorized_summary_df,
    ],
    ignore_index=True,
)

print("Fama-MacBeth regressions completed.")
print(
    "Raw-return months: "
    f"{len(fmb_raw_coefficients_df)}"
)
print(
    "Winsorized-return months: "
    f"{len(fmb_winsorized_coefficients_df)}"
)

In [ ]:
fmb_factor_results = (
    fmb_summary_df.loc[
        fmb_summary_df["variable"]
        != "const",
        [
            "specification",
            "variable",
            "number_of_months",
            "mean_monthly_coefficient",
            "annualized_coefficient",
            "newey_west_t_statistic",
            "newey_west_p_value",
            "positive_month_rate",
        ],
    ]
)

print("Fama-MacBeth factor results:")
print(
    fmb_factor_results
    .round(4)
    .to_string(index=False)
)

print("\nAverage monthly regression diagnostics:")

print(
    "Raw-return average adjusted R-squared: "
    f"{fmb_raw_coefficients_df['adjusted_r_squared'].mean():.4f}"
)

print(
    "Winsorized-return average adjusted R-squared: "
    f"{fmb_winsorized_coefficients_df['adjusted_r_squared'].mean():.4f}"
)

In [ ]:
raw_coefficients_output = (
    fmb_raw_coefficients_df.assign(
        specification="Raw Future Return"
    )
)

winsorized_coefficients_output = (
    fmb_winsorized_coefficients_df.assign(
        specification=(
            "Winsorized Future Return"
        )
    )
)

fmb_monthly_coefficients_output = pd.concat(
    [
        raw_coefficients_output,
        winsorized_coefficients_output,
    ],
    ignore_index=True,
)

fmb_monthly_coefficients_output.to_csv(
    TABLE_OUTPUT_DIR
    / "08_fama_macbeth_monthly_coefficients.csv",
    index=False,
)

fmb_summary_df.to_csv(
    TABLE_OUTPUT_DIR
    / "09_fama_macbeth_summary.csv",
    index=False,
)

print("Fama-MacBeth result files saved successfully.")

In [ ]:
SUBPERIODS = {
    "2005-2014": (
        pd.Timestamp("2005-01-31"),
        pd.Timestamp("2014-12-31"),
    ),
    "2015-2025": (
        pd.Timestamp("2015-01-31"),
        pd.Timestamp("2025-12-31"),
    ),
}

for period_name, (
    start_date,
    end_date,
) in SUBPERIODS.items():
    number_of_rows = factor_test_df.loc[
        factor_test_df["month"].between(
            start_date,
            end_date,
        )
    ].shape[0]

    print(
        f"{period_name}: "
        f"{number_of_rows:,} rows"
    )

In [ ]:
subperiod_fmb_summaries = []

for period_name, (
    start_date,
    end_date,
) in SUBPERIODS.items():

    period_data = factor_test_df.loc[
        factor_test_df["month"].between(
            start_date,
            end_date,
        )
    ].copy()

    period_coefficients = (
        run_monthly_cross_sectional_regressions(
            data=period_data,
            target_column=(
                "future_return_1m_winsorized"
            ),
            factor_columns=REGRESSION_FACTORS,
        )
    )

    period_summary = (
        summarize_fama_macbeth_coefficients(
            coefficient_data=(
                period_coefficients
            ),
            factor_columns=(
                REGRESSION_FACTORS
            ),
            specification_name=period_name,
        )
    )

    subperiod_fmb_summaries.append(
        period_summary
    )

subperiod_fmb_summary_df = pd.concat(
    subperiod_fmb_summaries,
    ignore_index=True,
)

subperiod_factor_results = (
    subperiod_fmb_summary_df.loc[
        subperiod_fmb_summary_df["variable"]
        != "const",
        [
            "specification",
            "variable",
            "number_of_months",
            "mean_monthly_coefficient",
            "annualized_coefficient",
            "newey_west_t_statistic",
            "newey_west_p_value",
            "positive_month_rate",
        ],
    ]
)

print("Subperiod Fama-MacBeth results:")
print(
    subperiod_factor_results
    .round(4)
    .to_string(index=False)
)

In [ ]:
subperiod_long_short_records = []

for period_name, (
    start_date,
    end_date,
) in SUBPERIODS.items():

    for factor_name in FACTOR_COLUMNS:

        for weighting_method, pivot_data in [
            (
                "Equal Weight",
                equal_weight_pivot,
            ),
            (
                "Value Weight",
                value_weight_pivot,
            ),
        ]:
            period_returns = pivot_data.loc[
                (
                    pivot_data["factor"]
                    == factor_name
                )
                & pivot_data["month"].between(
                    start_date,
                    end_date,
                ),
                "long_short_return",
            ]

            period_summary = (
                summarize_long_short_returns(
                    period_returns
                )
            )

            period_summary[
                "period"
            ] = period_name

            period_summary[
                "factor"
            ] = factor_name

            period_summary[
                "weighting_method"
            ] = weighting_method

            subperiod_long_short_records.append(
                period_summary
            )

subperiod_long_short_df = pd.DataFrame(
    subperiod_long_short_records
)

In [ ]:
selected_subperiod_results = (
    subperiod_long_short_df.loc[
        subperiod_long_short_df[
            "factor"
        ].isin([
            "factor_quality",
            "factor_investment",
            "multi_factor_score",
        ]),
        [
            "period",
            "factor",
            "weighting_method",
            "number_of_months",
            "annualized_return",
            "annualized_volatility",
            "annualized_sharpe",
            "newey_west_t_statistic",
            "newey_west_p_value",
            "positive_month_rate",
        ],
    ]
)

print("Selected subperiod long-short results:")
print(
    selected_subperiod_results
    .round(4)
    .to_string(index=False)
)

In [ ]:
subperiod_fmb_summary_df.to_csv(
    TABLE_OUTPUT_DIR
    / "10_subperiod_fama_macbeth_summary.csv",
    index=False,
)

subperiod_long_short_df.to_csv(
    TABLE_OUTPUT_DIR
    / "11_subperiod_long_short_summary.csv",
    index=False,
)

print("Subperiod validation results saved successfully.")